# Forget-MI BASELINE — Kaggle Notebook (EXP1-M)

> 🎯 **Mục đích**: Reproduce paper Forget-MI Table 2 trên Kaggle để có **số time/GPU đo trên cùng GPU với LoKU** (so sánh fair trong Chương 4 luận văn).
>
> **Support 2 datasets**: MIMIC-CXR (sẵn sàng) + Indiana University CXR (cần prep trước).

## Đặc điểm baseline vs LoKU
| | Baseline (notebook này) | LoKU (run.ipynb trên Colab) |
|---|---|---|
| Script | `training/forgetmi_partial.py` | `training/forgetmi_loku.py` |
| Trainable params | **100%** (full FT) | ~0.45% (LoRA + FILA) |
| Time/run | **~5h** (30 epochs) | ~12 min (8 epochs) |
| Loss | 4-loss Forget-MI gốc | 4-loss + IHL + FILA + distill |

## Cấu trúc notebook (15 cells)
| Cell | Mục đích | Phụ thuộc |
|---|---|---|
| 1 | Setup Kaggle env: clone repo, install deps | Mọi session |
| 2 | Verify input datasets + GPU (cả MIMIC + IU) | Sau Cell 1 |
| 3 | Define helpers + DATASETS config | **BẮT BUỘC trước Cell 4*** |
| **4a / 4b / 4c** | **MIMIC-CXR** — Train 3% / 6% / 10% (multi-seed) | Cần Kaggle Datasets MIMIC |
| **4d / 4e / 4f** | **IU-CXR** — Train 3% / 6% / 10% (multi-seed) | Cần IU prep xong + `enabled=True` |
| 5 | Bảng cross-dataset (MIMIC + IU × 3 forget% × Paper) | Sau Cell 4* |
| 6 | Push results lên GitHub (qua Kaggle Secrets) | Cần Kaggle Secrets |

## Setup Kaggle 1 lần (trước khi chạy)

### Bắt buộc (cho MIMIC)
1. **Kaggle Datasets** (Sidebar → + Add data → Upload):
   - `forget-mi-data` — chứa `metadata/` + `img_data/`
   - `forget-mi-models` — chứa `training_original_model/` + `model_retrained_3per/`
2. **Kaggle Secrets** (Add-ons → Secrets): `GITHUB_TOKEN`, `GIT_EMAIL`, `GIT_NAME`
3. **GPU**: Settings → Accelerator → **GPU T4 x2** (free 30h/tuần) hoặc **P100** nếu Pro

### Bổ sung (cho IU — khi đã prep xong theo THESIS_ROADMAP Section 13)
4. Tạo `config_baseline_iu_kaggle.yaml` (copy từ `config_baseline_kaggle.yaml`, đổi paths IU + `output_channel_encoding=binary`)
5. **Kaggle Datasets** thêm:
   - `forget-mi-data-iu`
   - `forget-mi-models-iu` (có `model_og_IU` + 3 `model_retrained_iu_Nper`)
6. Trong Cell 3, set `DATASETS['iu']['enabled'] = True`, rerun Cell 3

## Quota Kaggle free: 30h GPU/tuần

| Dataset | Forget% | Time | Cộng dồn |
|---|---|---|---|
| MIMIC 3% | × 3 seeds | ~15h | 15h |
| MIMIC 6% | × 3 seeds | ~15h | 30h (vừa khít tuần) |
| MIMIC 10% | × 3 seeds | ~15h | 45h (sang tuần sau) |
| IU 3-6-10% | × 2 seeds | ~30h | thêm 1 tuần |

→ Tổng đầy đủ 2 datasets: **~3 tuần** với free quota. Khuyến nghị: chạy MIMIC trước (xong sớm cho luận văn), IU sau."""

In [ ]:
# ====================================
# CELL 1: Setup Kaggle env
# ====================================
import os, sys, subprocess

WORK_DIR = "/kaggle/working"
REPO_URL = "https://github.com/nhnhu146/Forget-MI-LoKU.git"
REPO_NAME = "Forget-MI-LoKU"
REPO_DIR = f"{WORK_DIR}/{REPO_NAME}"

os.chdir(WORK_DIR)

# 1. Clone hoặc pull repo
if not os.path.exists(REPO_DIR):
    print(f"🔽 Clone {REPO_URL}")
    subprocess.run(['git', 'clone', REPO_URL], check=True)
else:
    print(f"🔄 Pull latest")
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/master'], check=True)

os.chdir(REPO_DIR)
print(f"📂 CWD: {os.getcwd()}")
subprocess.run(['git', 'log', '--oneline', '-1'])

# 2. Install deps (đúng version paper)
print("\n📦 Installing deps...")
!pip install -q pydicom scikit-image pyyaml
!pip install -q "transformers==4.38.0" "peft==0.10.0" "accelerate==0.27.0"

# 3. Check GPU
import torch
if torch.cuda.is_available():
    print(f"\n🟢 GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    print("\n🔴 KHÔNG CÓ GPU — baseline sẽ KHÔNG xong trong 12h Kaggle limit!")
    print("   → Settings → Accelerator → GPU T4 x2 → Save → Restart Session")

print("\n✅ Setup complete")

In [ ]:
# ====================================
# CELL 2: Verify input Kaggle datasets (CẢ MIMIC + IU)
# ====================================
import os

print("🔍 Kiểm tra input paths cho 2 datasets:\n")

# ---- Files chung (luôn cần — từ repo clone) ----
COMMON = {
    "Synonyms":     "./data_splits/Synonyms.csv",
    "Forget 3%":    "./data_splits/forget_set_3per.csv",
    "Forget 6%":    "./data_splits/forget_set_6per.csv",
    "Forget 10%":   "./data_splits/forget_set_10per.csv",
}

# ---- MIMIC dataset ----
# NOTE: Cả 2 Kaggle Datasets có nested wrappers từ zip extraction
# Cấu trúc thực tế (confirmed):
#   /kaggle/input/forget-mi-data/
#   ├── data/{metadata, img_data, text_data}/
#   └── data_splits/...
#   /kaggle/input/forget-mi-models/
#   ├── base_model/training_original_model/      ← 2 levels nested
#   └── retrained_model/model_retrained_3per/    ← 2 levels nested
MIMIC = {
    "Base model":      "/kaggle/input/forget-mi-models/base_model/training_original_model/pytorch_model.bin",
    "Retrained 3%":    "/kaggle/input/forget-mi-models/retrained_model/model_retrained_3per/pytorch_model.bin",
    "Text metadata":   "/kaggle/input/forget-mi-data/data/metadata",
    "Image data":      "/kaggle/input/forget-mi-data/data/img_data",
    "MIMIC split CSV": "./data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv",
}

# ---- IU dataset (placeholder — giả sử cùng pattern khi upload sau) ----
IU = {
    "IU base model":   "/kaggle/input/forget-mi-models-iu/base_model/model_og_IU/pytorch_model.bin",
    "IU retrained 3%": "/kaggle/input/forget-mi-models-iu/retrained_model/model_retrained_iu_3per/pytorch_model.bin",
    "IU text data":    "/kaggle/input/forget-mi-data-iu/data/metadata",
    "IU image data":   "/kaggle/input/forget-mi-data-iu/data/img_data",
    "IU split CSV":    "./data_iu/splits/train_test_split_iu.csv",
    "IU forget 3%":    "./data_iu/splits/forget_set_3per_iu.csv",
}


def _verify_group(title, checks, required=True):
    print(f"━━ {title} ━━")
    all_ok = True
    for name, path in checks.items():
        if os.path.exists(path):
            if os.path.isdir(path):
                n = len(os.listdir(path))
                print(f"  ✅ {name:<18} {path}  ({n} items)")
            else:
                size = os.path.getsize(path) / 1e6
                print(f"  ✅ {name:<18} {path}  ({size:.1f} MB)")
        else:
            icon = "❌" if required else "⚠️ "
            print(f"  {icon} {name:<18} {path}  KHÔNG TỒN TẠI")
            if required:
                all_ok = False
    return all_ok


common_ok = _verify_group("FILES CHUNG (từ repo clone)", COMMON, required=True)
print()
mimic_ok  = _verify_group("MIMIC-CXR (BẮT BUỘC cho Cell 4a/4b/4c)", MIMIC, required=True)
print()
iu_ok     = _verify_group("INDIANA UNIVERSITY CXR (cần cho Cell 4d/4e/4f)", IU, required=False)

print()
print("━" * 70)
if common_ok and mimic_ok:
    print("✅ MIMIC-CXR sẵn sàng — chạy được Cell 4a/4b/4c")
else:
    print("❌ MIMIC chưa OK — kiểm tra File Browser bên trái:")
    print("   1. Sidebar trái → Data → expand forget-mi-data/ và forget-mi-models/")
    print("   2. Confirm có: forget-mi-data/data/{metadata,img_data}/")
    print("                  forget-mi-models/base_model/training_original_model/")
    print("                  forget-mi-models/retrained_model/model_retrained_3per/")
    print("   3. Nếu cấu trúc khác → sửa paths trong config_baseline_kaggle.yaml")
print()
if iu_ok:
    print("✅ IU-CXR sẵn sàng — Cell 3 cần set DATASETS['iu']['enabled']=True")
else:
    print("⚠️  IU-CXR chưa sẵn sàng (bình thường nếu chưa prep)")
    print("    → Cell 4d/4e/4f sẽ tự skip — không crash notebook")

In [ ]:
# ====================================
# CELL 3: Helpers cho baseline multi-seed (BẮT BUỘC trước Cell 4*)
# ====================================
# Support 2 datasets: MIMIC-CXR (sẵn sàng) + Indiana University CXR (cần prep).
# Để enable IU: set `DATASETS['iu']['enabled'] = True` sau khi đã preprocess
# theo Section 13 của THESIS_ROADMAP.md + upload Kaggle Datasets tương ứng.

import os, numpy as np, pandas as pd
from datetime import datetime

# CSV chung cho cả 2 datasets (column 'dataset' phân biệt)
CSV_PATH = "/kaggle/working/results_summary.csv"

# Paper Forget-MI Table 1 reference numbers (chỉ cho MIMIC-CXR)
PAPER_REF_MIMIC = {
    3:  {"MIA_paper": 0.571, "Df_AUC": 0.735, "Df_F1": 0.393, "Dt_AUC": 0.625, "Dt_F1": 0.250, "Time_h": 5.0},
    6:  {"MIA_paper": 0.615, "Df_AUC": 0.654, "Df_F1": 0.328, "Dt_AUC": 0.599, "Dt_F1": 0.270, "Time_h": 5.0},
    10: {"MIA_paper": 0.810, "Df_AUC": 0.656, "Df_F1": 0.313, "Dt_AUC": 0.565, "Dt_F1": 0.252, "Time_h": 5.0},
}

# Dataset config — extensible cho IU và datasets khác
# ⚠️ Cấu trúc thực tế Kaggle Datasets (confirmed qua File Browser):
#   /kaggle/input/forget-mi-data/
#   ├── data/                                    ← wrapper từ data.zip
#   │   ├── metadata/
#   │   ├── img_data/
#   │   └── text_data/
#   └── data_splits/                             ← repo clone cũng có → notebook dùng từ repo
#
#   /kaggle/input/forget-mi-models/
#   ├── base_model/                              ← wrapper từ base_model.zip
#   │   └── training_original_model/
#   │       └── pytorch_model.bin + config.json + vocab.txt + ...
#   └── retrained_model/                         ← wrapper từ retrained_model.zip
#       └── model_retrained_3per/
#           └── pytorch_model.bin + ...
DATASETS = {
    'mimic': {
        'enabled': True,                                # ✅ sẵn sàng (dataset gốc paper)
        'name': 'MIMIC-CXR',
        'config_file': 'config_baseline_kaggle.yaml',
        # data_root = thư mục chứa metadata/, img_data/, text_data/ TRỰC TIẾP
        'data_root': '/kaggle/input/forget-mi-data/data',
        'models_root': '/kaggle/input/forget-mi-models',
        # 2 wrappers nested vì 2 zip extract riêng
        'base_model_subdir': 'base_model/training_original_model',
        'gold_retrained': {
            3:  'retrained_model/model_retrained_3per',
            6:  None,
            10: None,
        },
        'data_split': './data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv',
        'forget_set_template': './data_splits/forget_set_{}per.csv',
        'paper_ref': PAPER_REF_MIMIC,
        'extra_overrides': '',
    },
    'iu': {
        'enabled': False,
        'name': 'Indiana University CXR',
        'config_file': 'config_baseline_iu_kaggle.yaml',
        # Giả sử cùng pattern: zip extract → có wrapper folder cùng tên
        'data_root': '/kaggle/input/forget-mi-data-iu/data',
        'models_root': '/kaggle/input/forget-mi-models-iu',
        'base_model_subdir': 'base_model/model_og_IU',
        'gold_retrained': {
            3:  'retrained_model/model_retrained_iu_3per',
            6:  'retrained_model/model_retrained_iu_6per',
            10: 'retrained_model/model_retrained_iu_10per',
        },
        'data_split': './data_iu/splits/train_test_split_iu.csv',
        'forget_set_template': './data_iu/splits/forget_set_{}per_iu.csv',
        'paper_ref': None,
        'extra_overrides': 'output_channel_encoding=binary',
    },
}

METRIC_DEFS = [
    ('MIA',                 'MIA_persample',   None,         '↓'),
    ('MIA_paper',           'MIA_paper',       'MIA_paper',  '↓'),
    ('forget_ce',           'forget_ce',       None,         '·'),
    ('test_ce',             'test_ce',         None,         '·'),
    ('Df_AUC',              'Forget AUC',      'Df_AUC',     '↓'),
    ('Df_F1',               'Forget Mac-F1',   'Df_F1',      '↓'),
    ('Dt_AUC',              'Test AUC',        'Dt_AUC',     '↑'),
    ('Dt_F1',               'Test Mac-F1',     'Dt_F1',      '↑'),
    ('dist_vs_re',          '1 − CosSim',      None,         '↓'),
    ('unlearn_time_hours',  'Time (h)',        'Time_h',     '↓'),
    ('gpu_peak_GB',         'GPU peak (GB)',   None,         '·'),
    ('trainable_ratio',     'Trainable ratio', None,         '·'),
]


def _check_dataset_ready(dataset_key):
    """Verify dataset config + Kaggle input paths đã sẵn sàng."""
    ds = DATASETS[dataset_key]
    if not ds['enabled']:
        return False, f"DATASETS['{dataset_key}']['enabled']=False — set =True khi đã prep xong"
    if not os.path.exists(ds['config_file']):
        return False, f"Config file không tồn tại: {ds['config_file']}"
    # Check base model path (sau khi đã apply wrappers)
    base_model = os.path.join(ds['models_root'], ds['base_model_subdir'])
    if not os.path.isdir(base_model):
        return False, f"Pretrained model không tồn tại: {base_model}/  → Check Kaggle Dataset structure"
    # Check pytorch_model.bin file thật
    if not os.path.exists(os.path.join(base_model, 'pytorch_model.bin')):
        return False, f"pytorch_model.bin không tồn tại trong {base_model}/  → wrappers sai?"
    # Check data root + subdirs
    if not os.path.isdir(ds['data_root']):
        return False, f"Data root không tồn tại: {ds['data_root']}/  → Check Kaggle Dataset structure"
    for sub in ('metadata', 'img_data'):
        if not os.path.isdir(os.path.join(ds['data_root'], sub)):
            return False, (f"Không tìm thấy {ds['data_root']}/{sub}/. "
                           f"Cấu trúc Kaggle Dataset khác config — kiểm tra File Browser bên trái.")
    return True, "ready"


def _check_gold(dataset_key, forget_pct):
    """Check gold retrained model cho dataset + forget% cụ thể."""
    ds = DATASETS[dataset_key]
    subdir = ds['gold_retrained'].get(forget_pct)
    if subdir is None:
        return None, False
    path = os.path.join(ds['models_root'], subdir)
    return path, os.path.isdir(path)


def _seed_done(dataset_key, forget_pct, seed):
    """Check (dataset, forget_pct, seed) đã có trong CSV chưa."""
    if not os.path.exists(CSV_PATH):
        return False
    try:
        df = pd.read_csv(CSV_PATH)
    except Exception:
        return False
    for col in ('forget_pct', 'seed', 'method'):
        if col not in df.columns:
            return False
    mask = (df['forget_pct'].astype(str).str.contains(f"_{forget_pct}per")
            & (df['seed'] == seed)
            & (df['method'] == 'baseline_partial'))
    if 'dataset' in df.columns:
        mask = mask & (df['dataset'] == dataset_key)
    return bool(mask.any())


def run_baseline_multiseed(forget_pct, dataset='mimic', seeds=(42, 123, 7), force_redo=False):
    """Chạy baseline forgetmi_partial.py multi-seed cho 1 (dataset, forget%) cụ thể.

    Args:
        forget_pct (int): 3, 6, or 10
        dataset (str): 'mimic' hoặc 'iu'
        seeds (tuple): random seeds (default 3 seeds chính thức)
        force_redo (bool): chạy lại dù seed đã có trong CSV
    """
    assert forget_pct in (3, 6, 10), f"forget_pct phải 3/6/10, nhận {forget_pct}"
    assert dataset in DATASETS, f"dataset phải trong {list(DATASETS)}, nhận {dataset!r}"

    ds = DATASETS[dataset]
    ready, msg = _check_dataset_ready(dataset)
    if not ready:
        print(f"❌ Dataset '{dataset}' chưa sẵn sàng: {msg}")
        print(f"   → Xem THESIS_ROADMAP.md Section 13 (IU prep) hoặc KAGGLE_SETUP.md")
        return

    forget_csv = ds['forget_set_template'].format(forget_pct)
    retrained_path, has_gold = _check_gold(dataset, forget_pct)
    exp_name = f"baseline_{dataset}_{forget_pct}per"
    paper_ref = ds['paper_ref'][forget_pct] if ds['paper_ref'] else None
    base_model = os.path.join(ds['models_root'], ds['base_model_subdir'])

    # ---- Banner ----
    print(f"\n{'#'*78}")
    print(f"# 🎯 BASELINE Forget-MI — {ds['name']} — FORGET {forget_pct}% — seeds {list(seeds)}")
    print(f"# Config         : {ds['config_file']}")
    print(f"# Data root      : {ds['data_root']}")
    print(f"# Base model     : {base_model}")
    print(f"# Forget CSV     : {forget_csv}")
    print(f"# Run name       : {exp_name}")
    if has_gold:
        print(f"# Gold retrained : ✅ {retrained_path}")
    else:
        print(f"# Gold retrained : ❌ N/A cho {dataset} {forget_pct}% → 1−CosSim KHÔNG hợp lệ")
    print(f"# Paper ref      : {'✅ available' if paper_ref else '❌ N/A (IU không có paper baseline)'}")
    print(f"# ⏱  Dự kiến     : ~{5 * len(seeds)}h tổng (paper báo 5h/run)")
    print(f"{'#'*78}\n")

    if not os.path.exists(forget_csv):
        raise FileNotFoundError(f"Forget set không tồn tại: {forget_csv}")

    # ---- Build override ----
    # data_root + base_model_subdir + gold_retrained đã chứa wrappers nested
    OVR_parts = [
        f"forget_set_path={forget_csv}",
        f"id={exp_name}",
        f"base_model_path={base_model}",
        f"bert_pretrained_dir={base_model}",
        f"text_data_dir={ds['data_root']}/metadata",
        f"img_data_dir={ds['data_root']}/img_data",
        f"data_split_path={ds['data_split']}",
        "output_dir=/kaggle/working/baseline_output",
    ]
    if has_gold:
        OVR_parts.append(f"retrained_model_path={retrained_path}")
    if ds['extra_overrides']:
        OVR_parts.append(ds['extra_overrides'])
    OVR = ",".join(OVR_parts)

    HYPOTHESIS = (f"Baseline Forget-MI tại {ds['name']} {forget_pct}%, Kaggle reproduce. "
                  f"Multi-seed → mean±std cho luận văn Chương 4.")

    # ---- Loop seeds ----
    for i, s in enumerate(seeds):
        if not force_redo and _seed_done(dataset, forget_pct, s):
            print(f"⏭️  SEED {s} đã có trong CSV cho {dataset} {forget_pct}% — bỏ qua\n")
            continue
        print(f"\n{'='*60}\n🎲 SEED {s}  ({i+1}/{len(seeds)})  ▸ {dataset.upper()} {forget_pct}%\n{'='*60}")
        cmd = (f'PYTHONPATH=. WANDB_MODE=disabled python training/forgetmi_partial.py '
               f'--config {ds["config_file"]} --fresh --seed {s} '
               f'--override "{OVR}" '
               f'--exp {exp_name}_seed{s} --hypothesis "{HYPOTHESIS}"')
        get_ipython().system(cmd)

    # ---- Tag dataset + merge CSV ----
    src_csv = "/kaggle/working/baseline_output/results_summary.csv"
    if os.path.exists(src_csv):
        df_src = pd.read_csv(src_csv)
        if 'dataset' not in df_src.columns:
            df_src['dataset'] = dataset
        if os.path.exists(CSV_PATH):
            df_main = pd.read_csv(CSV_PATH)
            df_merged = pd.concat([df_main, df_src], ignore_index=True)
            df_merged = df_merged.drop_duplicates(
                subset=['dataset', 'forget_pct', 'seed'], keep='last'
            ) if 'dataset' in df_merged.columns else df_merged
            df_merged.to_csv(CSV_PATH, index=False)
        else:
            df_src.to_csv(CSV_PATH, index=False)
        os.remove(src_csv)

    aggregate_baseline_summary(dataset, forget_pct, seeds, exp_name, paper_ref, has_gold)


def aggregate_baseline_summary(dataset_key, forget_pct, seeds, exp_name, paper_ref, has_gold):
    """In bảng mean±std + so paper (nếu có) + lưu MD summary."""
    if not os.path.exists(CSV_PATH):
        print(f"❌ {CSV_PATH} không tồn tại. Bỏ qua aggregate.")
        return

    df = pd.read_csv(CSV_PATH)
    mask = (df['forget_pct'].astype(str).str.contains(f"_{forget_pct}per")
            & df['seed'].isin(seeds))
    if 'method' in df.columns:
        mask = mask & (df['method'] == 'baseline_partial')
    if 'dataset' in df.columns:
        mask = mask & (df['dataset'] == dataset_key)
    df = df[mask]
    if df.empty:
        print(f"❌ Không có rows cho {dataset_key} {forget_pct}% + seeds={list(seeds)}.")
        return

    if 'timestamp' in df.columns:
        df = df.sort_values('timestamp').drop_duplicates(subset=['seed'], keep='last')

    ds = DATASETS[dataset_key]
    print(f"\n{'='*100}")
    print(f"📊 BASELINE SUMMARY — {ds['name']} — FORGET {forget_pct}% — seeds={list(seeds)}")
    if not has_gold:
        print(f"⚠️  1−CosSim KHÔNG hợp lệ ({dataset_key} {forget_pct}%: no gold retrained)")
    if not paper_ref:
        print(f"ℹ️  Paper ref không có cho {ds['name']} → bỏ cột Δ vs paper")
    print(f"{'='*100}")

    hdr_paper = f"{'paper':>10}{'Δ vs paper':>12}" if paper_ref else ""
    hdr = (f"{'Metric':<20}" + "".join(f"{f'seed{s}':>10}" for s in seeds)
           + f"{'mean ± std':>18}" + hdr_paper)
    print(hdr)
    print("-" * len(hdr))

    md_cols = ["Metric"] + [f"seed {s}" for s in seeds] + ["**mean ± std**"]
    if paper_ref:
        md_cols += ["Paper", "Δ vs paper"]
    md_rows = ["| " + " | ".join(md_cols) + " |",
               "|" + "---|" * len(md_cols)]

    for csv_key, label, paper_key, direction in METRIC_DEFS:
        if csv_key not in df.columns:
            continue
        vals = []
        for s in seeds:
            sub = df[df['seed'] == s]
            if sub.empty: continue
            try:
                v = float(sub[csv_key].iloc[-1])
            except Exception:
                continue
            if v != v: continue
            vals.append(v)
        if not vals:
            continue
        m, sd = float(np.mean(vals)), float(np.std(vals))
        invalid_marker = " ⚠️" if (csv_key == 'dist_vs_re' and not has_gold) else ""

        if paper_ref and paper_key and paper_key in paper_ref:
            p_val = paper_ref[paper_key]
            delta = m - p_val
            good = (direction == '↓' and delta < 0) or (direction == '↑' and delta > 0)
            sym = "✅" if good else ("❌" if direction in ('↓','↑') else "·")
            paper_console = f"{p_val:>10.3f}{sym}{delta:+.3f}".rjust(22)
            md_paper = f"{p_val:.3f}"
            md_delta = f"{delta:+.3f}"
        else:
            paper_console = "" if not paper_ref else f"{'—':>10}{'—':>12}"
            md_paper = "—"
            md_delta = "—"

        cell_vals = "".join(f"{v:>10.3f}" for v in vals)
        cell_vals += " " * (10 * (len(seeds) - len(vals)))
        ms_str = f"{m:>10.3f}±{sd:.3f}"
        print(f"{label+invalid_marker:<20}{cell_vals}{ms_str:>18}{paper_console}")

        md_vals_str = " | ".join(f"{v:.3f}" for v in vals) + " | " * (len(seeds) - len(vals))
        md_row = f"| {label}{invalid_marker} | {md_vals_str} | **{m:.3f} ± {sd:.3f}** |"
        if paper_ref:
            md_row += f" {md_paper} | {md_delta} |"
        md_rows.append(md_row)

    os.makedirs("experiments", exist_ok=True)
    out_md = f"experiments/summary_baseline_{dataset_key}_{forget_pct}per_multiseed.md"
    paper_line = ""
    if paper_ref:
        paper_line = (f"\n**Paper Forget-MI ({forget_pct}%)**: "
                      f"MIA={paper_ref['MIA_paper']} | Df_AUC={paper_ref['Df_AUC']} | "
                      f"Df_F1={paper_ref['Df_F1']} | Dt_AUC={paper_ref['Dt_AUC']} | "
                      f"Dt_F1={paper_ref['Dt_F1']} | Time≈{paper_ref['Time_h']}h\n")
    body = "\n".join([
        f"# BASELINE Forget-MI Multi-seed — {ds['name']} — FORGET {forget_pct}%",
        "",
        f"_Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} on Kaggle_",
        "",
        f"**Dataset**     : {ds['name']} (`{dataset_key}`)",
        f"**Config**      : `{ds['config_file']}`",
        f"**Seeds**       : {list(seeds)}",
        f"**Gold retrain**: {'✅ available' if has_gold else '❌ N/A — 1−CosSim không hợp lệ'}",
        "",
        *md_rows,
        paper_line,
    ])
    with open(out_md, "w", encoding="utf-8") as f:
        f.write(body)
    print(f"\n💾 Summary MD: {out_md}")


# ---- Print readiness status ----
print("✅ Helpers đã định nghĩa: run_baseline_multiseed(forget_pct, dataset=...), aggregate_baseline_summary()")
print(f"   CSV path: {CSV_PATH}\n")
print("📋 Dataset readiness:")
for k, ds in DATASETS.items():
    status = "✅ ENABLED" if ds['enabled'] else "❌ DISABLED (set enabled=True khi prep xong)"
    print(f"   • {k:<6} ({ds['name']}): {status}")
    print(f"            data_root : {ds['data_root']}")
    print(f"            base_model: {os.path.join(ds['models_root'], ds['base_model_subdir'])}")
    if ds['enabled']:
        ok, msg = _check_dataset_ready(k)
        if not ok:
            print(f"            ⚠️  {msg}")
        else:
            print(f"            ✅ ready")
print()
print("Usage:")
print("   run_baseline_multiseed(forget_pct=3, dataset='mimic')   # default")
print("   run_baseline_multiseed(forget_pct=6, dataset='iu')      # khi IU sẵn sàng")

---

## 🎯 Training cells — chạy theo nhu cầu

⚠️ **Quota Kaggle free: 30h GPU/tuần**. Mỗi forget% × 3 seeds ≈ 15h.

Khuyến nghị thứ tự (theo priority luận văn):
1. **Cell 4a (MIMIC 3%)** — quan trọng nhất (so paper Table 2 Unimodal 3%)
2. **Cell 4c (MIMIC 10%)** — case khó nhất paper
3. **Cell 4b (MIMIC 6%)** — case trung gian
4. **Cell 4d-4f (IU 3/6/10%)** — generalization check (chỉ khi IU prep xong)

### 📊 MIMIC-CXR — Cells 4a, 4b, 4c

In [ ]:
# ====================================
# CELL 4a: MIMIC-CXR — FORGET 3% (multi-seed)
# ====================================
# Expected: paper Table 2 Unimodal 3% → MIA=0.571, Df_AUC=0.735, Dt_AUC=0.625, Time~5h
# Multi-seed sẽ ra số mean±std (paper chỉ báo 1 seed)

run_baseline_multiseed(forget_pct=3, dataset='mimic', seeds=(42, 123, 7))

In [ ]:
# ====================================
# CELL 4b: MIMIC-CXR — FORGET 6% (multi-seed)
# ====================================
# Expected: paper Table 2 Multimodal 6% → MIA=0.615, Df_AUC=0.654, Dt_AUC=0.599
# ⚠️ Chưa có gold retrained cho 6% → 1−CosSim KHÔNG hợp lệ

run_baseline_multiseed(forget_pct=6, dataset='mimic', seeds=(42, 123, 7))

In [ ]:
# ====================================
# CELL 4c: MIMIC-CXR — FORGET 10% (multi-seed)
# ====================================
# Expected: paper Table 2 Retention 10% → MIA=0.810, Df_AUC=0.656, Dt_AUC=0.565
# ⚠️ Chưa có gold retrained cho 10% → 1−CosSim KHÔNG hợp lệ

run_baseline_multiseed(forget_pct=10, dataset='mimic', seeds=(42, 123, 7))

---

## 🔬 Indiana University CXR — generalization check

⚠️ **Yêu cầu trước khi chạy cells dưới**:
1. Preprocess IU-CXR theo Section 13 của `THESIS_ROADMAP.md` (download Open-i, parse XML, sinh label binary normal/abnormal, tạo splits, retrain models)
2. Tạo file `config_baseline_iu_kaggle.yaml` (copy `config_baseline_kaggle.yaml`, đổi `output_channel_encoding=binary` + paths IU)
3. Upload 2 Kaggle Datasets:
   - `forget-mi-data-iu` (metadata + img_data IU)
   - `forget-mi-models-iu` (model_og_IU + model_retrained_iu_*per)
4. Set `DATASETS['iu']['enabled'] = True` trong Cell 3 → rerun Cell 3 → chạy cells dưới

Nếu chưa làm xong, cells này sẽ chỉ in thông báo "dataset chưa sẵn sàng" và skip — không crash notebook.

In [ ]:
# ====================================
# CELL 4d: IU-CXR — FORGET 3% (multi-seed)
# ====================================
# Task: binary normal/abnormal (proxy cho edema multi-class của MIMIC).
# Không có paper ref cho IU → so với chính baseline IU ở các forget% khác để xem trend.
# Cell tự skip nếu IU chưa enable trong DATASETS.

run_baseline_multiseed(forget_pct=3, dataset='iu', seeds=(42, 123))

In [ ]:
# ====================================
# CELL 4e: IU-CXR — FORGET 6% (multi-seed)
# ====================================
# Cell tự skip nếu IU chưa enable hoặc retrained 6% chưa có.

run_baseline_multiseed(forget_pct=6, dataset='iu', seeds=(42, 123))

In [ ]:
# ====================================
# CELL 4f: IU-CXR — FORGET 10% (multi-seed)
# ====================================
# Cell tự skip nếu IU chưa enable hoặc retrained 10% chưa có.

run_baseline_multiseed(forget_pct=10, dataset='iu', seeds=(42, 123))

In [ ]:
# ====================================
# CELL 5: BẢNG CROSS-DATASET — Baseline trên MIMIC + IU (cho Chương 4 luận văn)
# ====================================
# Đọc CSV chung → in bảng so sánh 2 datasets × 3 forget% × Paper ref (chỉ MIMIC).
# Chạy sau khi đã hoàn thành ≥ 1 cell 4* (MIMIC hoặc IU).

import os, numpy as np, pandas as pd
from datetime import datetime

if not os.path.exists(CSV_PATH):
    print(f"❌ {CSV_PATH} không tồn tại. Chạy ít nhất 1 cell 4* trước.")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"📋 Tổng rows: {len(df)}")
    for col in ('dataset', 'method', 'forget_pct'):
        if col in df.columns:
            print(f"📋 {col:<12}: {df[col].unique().tolist()}")
    print()

    show_metrics = [
        ('MIA_paper',          'MIA',         '↓'),
        ('Df_AUC',             'Df_AUC',      '↓'),
        ('Df_F1',              'Df_F1',       '↓'),
        ('Dt_AUC',             'Dt_AUC',      '↑'),
        ('Dt_F1',              'Dt_F1',       '↑'),
        ('dist_vs_re',         '1−CosSim',    '↓'),
        ('unlearn_time_hours', 'Time(h)',     '↓'),
    ]

    md = [
        "# Bảng baseline Kaggle — Cross-dataset (MIMIC + IU)",
        "",
        f"_Auto-generated từ `{CSV_PATH}` — {datetime.now().strftime('%Y-%m-%d %H:%M')}_",
        "",
        "Bảng chứa BASELINE Forget-MI trên Kaggle cho cả 2 datasets.",
        "Để so với LoKU, merge CSV này với CSV từ Colab.",
        "",
    ]

    header = "| Dataset | Forget% | Method | " + " | ".join(m[1] for m in show_metrics) + " |"
    sep = "|---|---|---|" + "---|" * len(show_metrics)
    md += [header, sep]
    print("=" * 115); print(header); print("=" * 115)

    paper_key_map = {'MIA_paper': 'MIA_paper', 'Df_AUC': 'Df_AUC', 'Df_F1': 'Df_F1',
                     'Dt_AUC': 'Dt_AUC', 'Dt_F1': 'Dt_F1', 'unlearn_time_hours': 'Time_h'}

    for dataset_key in ['mimic', 'iu']:
        ds = DATASETS[dataset_key]
        # Filter rows for this dataset
        if 'dataset' in df.columns:
            sub_ds = df[df['dataset'] == dataset_key]
        else:
            # Fallback: cũ không có column dataset → assume mimic
            sub_ds = df if dataset_key == 'mimic' else df.iloc[0:0]

        if sub_ds.empty and not ds['enabled']:
            # IU chưa enable + chưa có data → skip section
            continue

        # Section divider
        section_row = f"| **{ds['name']}** | | | " + " | ".join("" for _ in show_metrics) + " |"
        md.append(section_row)
        print(f"\n━━━ {ds['name']} ━━━")

        for pct in [3, 6, 10]:
            sub_pct = sub_ds[sub_ds['forget_pct'].astype(str).str.contains(f"_{pct}per")] if not sub_ds.empty else sub_ds
            if 'method' in sub_pct.columns:
                sub_pct = sub_pct[sub_pct['method'] == 'baseline_partial']

            # ----- Paper row (chỉ MIMIC) -----
            if ds['paper_ref'] and pct in ds['paper_ref']:
                paper_row = f"| {ds['name']} | {pct}% | Paper |"
                for csv_k, _, _ in show_metrics:
                    if csv_k in paper_key_map and paper_key_map[csv_k] in ds['paper_ref'][pct]:
                        paper_row += f" {ds['paper_ref'][pct][paper_key_map[csv_k]]:.3f} |"
                    else:
                        paper_row += " — |"
                md.append(paper_row); print(paper_row)

            # ----- Baseline row -----
            if sub_pct.empty:
                row = f"| {ds['name']} | {pct}% | _(chưa chạy)_ |" + " — |" * len(show_metrics)
            else:
                if 'timestamp' in sub_pct.columns:
                    sub_pct = sub_pct.sort_values('timestamp').drop_duplicates(subset=['seed'], keep='last')
                n = sub_pct['seed'].nunique()
                row = f"| {ds['name']} | {pct}% | **Baseline (n={n})** |"
                for csv_k, _, _ in show_metrics:
                    if csv_k not in sub_pct.columns:
                        row += " — |"; continue
                    vals = sub_pct[csv_k].dropna().values
                    if len(vals) == 0:
                        row += " — |"; continue
                    mv, sd = float(np.mean(vals)), float(np.std(vals))
                    # mark CosSim invalid khi no gold
                    no_gold_pct = ds['gold_retrained'].get(pct) is None
                    mark = "⚠️" if (csv_k == 'dist_vs_re' and no_gold_pct) else ""
                    row += f" {mv:.3f}±{sd:.3f}{mark} |"
            md.append(row); print(row)

            # ----- Δ row (chỉ khi có paper + baseline) -----
            if (ds['paper_ref'] and pct in ds['paper_ref']
                    and not sub_pct.empty):
                delta_row = f"| {ds['name']} | {pct}% | Δ (LoKU−paper) |"
                for csv_k, _, direction in show_metrics:
                    if (csv_k not in sub_pct.columns
                            or csv_k not in paper_key_map
                            or paper_key_map[csv_k] not in ds['paper_ref'][pct]):
                        delta_row += " — |"; continue
                    vals = sub_pct[csv_k].dropna().values
                    if len(vals) == 0:
                        delta_row += " — |"; continue
                    mv = float(np.mean(vals))
                    p = ds['paper_ref'][pct][paper_key_map[csv_k]]
                    d = mv - p
                    good = (direction == '↓' and d < 0) or (direction == '↑' and d > 0)
                    sym = "✅" if good else "❌"
                    delta_row += f" {sym}{d:+.3f} |"
                md.append(delta_row); print(delta_row)
            md.append("")  # spacing
            print("-" * 115)

    out_md = "experiments/bang_baseline_kaggle_cross_dataset.md"
    with open(out_md, "w", encoding="utf-8") as f:
        f.write("\n".join(md))
    print(f"\n💾 Bảng cross-dataset saved: {out_md}")
    print(f"   → Dùng cho Chương 4.9-4.11 luận văn (so sánh MIMIC vs IU)")

In [ ]:
# ====================================
# CELL 6: Push results lên GitHub (Kaggle Secrets)
# ====================================
# Cần setup 1 lần: Add-ons → Secrets → Add secret:
#   GITHUB_TOKEN = ghp_xxx (scope: repo)
#   GIT_EMAIL    = your@email.com
#   GIT_NAME     = Your Name

import os, subprocess

GITHUB_REPO = "nhnhu146/Forget-MI-LoKU"
BRANCH = "master"

# Load credentials từ Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    TOKEN = secrets.get_secret('GITHUB_TOKEN')
    EMAIL = secrets.get_secret('GIT_EMAIL')
    NAME = secrets.get_secret('GIT_NAME')
    print("🔑 Credentials từ Kaggle Secrets ✅")
except Exception as e:
    print(f"⚠️  Không load được Kaggle Secrets ({e}).")
    print("   Setup: Add-ons → Secrets → Add: GITHUB_TOKEN, GIT_EMAIL, GIT_NAME")
    TOKEN = EMAIL = NAME = None

if TOKEN and EMAIL and NAME:
    !git config user.email "{EMAIL}"
    !git config user.name "{NAME}"
    !git remote set-url origin https://{TOKEN}@github.com/{GITHUB_REPO}.git
    !git pull --rebase origin {BRANCH} 2>&1 | tail -5
    !git add experiments/ 2>/dev/null

    changes = !git diff --cached --name-only
    if changes and any(c.strip() for c in changes):
        print("\n📦 Files commit:")
        for f in changes:
            if f.strip():
                print(f"   - {f}")
        msg = "baseline kaggle: auto-tracked multi-seed results"
        !git commit -m "{msg}"
        !git push origin {BRANCH}
        print(f"\n✅ Pushed: https://github.com/{GITHUB_REPO}/tree/{BRANCH}/experiments")
    else:
        print("ℹ️  Không có file mới để commit.")
else:
    print("⚠️  Thiếu credentials — skip push.")
    print("   Tải kết quả thủ công: /kaggle/working/results_summary.csv + experiments/*.md")